In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats


: 

In [ ]:
index_names = ['unit_number', 'time_cycles']
operational_settings = ['setting_1', 'setting_2', 'setting_3']

sensor_names = [f'S_{i}' for i in range(1,22)]
col_names = index_names + operational_settings + sensor_names

print(f"Column names defined. Total columns: {len(col_names)}")

In [ ]:
train_df  = pd.read_csv(f'train_FD001.txt', sep=r'\s+', header=None, names=col_names)
test_df  = pd.read_csv(f'test_FD001.txt', sep=r'\s+', header=None, names=col_names)
rul_df  = pd.read_csv(f'RUL_FD001.txt', sep=r'\s+', header=None, names=col_names)

pd.set_option("display.max_columns", None)
train_df

In [ ]:
train_df.isnull().sum()

In [ ]:
print(train_df.dtypes)

In [ ]:
train_df.describe()

In [ ]:
train_df.groupby('unit_number')['time_cycles'].max()

Il y a 100 moteur différents avec des temps de cylces varié. On en conclu qu'il n'y a pas une durée standard de vie pour les moteurs


# **choix des capteurs**

## **Etape 1**  : Exploration visuelle et ration pente/bruit

In [ ]:
fig, axes = plt.subplots(7, 3, figsize=(15, 20))
axes = axes.flatten()

moteur1 = train_df[train_df['unit_number'] == 1]

for i, sensor in enumerate(sensor_names):
    axes[i].plot(moteur1['time_cycles'], moteur1[sensor])
    axes[i].set_title(sensor)
    axes[i].set_xlabel('Cycle')

plt.tight_layout()
plt.show()

On remarque par ces visualisations que tous les capteurs n'apportent pas une information les capteurs (S_19, S_18, S_16, S_10, S_8, S_9, S_6, S_5,et S_1), ils seront assez naturellement retiré des données à analyser. Par ailleurs une vérification statistique s'impose  

In [ ]:
moteur1[sensor_names].std().sort_values()

Le cacul de l'écart-type le confirme, les capteurs `S_19, S_18, S_16, S_10, S_6, S_5, S_1`, n'apportent aucune information d'autant plus que leurs écart-types sont presque égles à 0. <br>
Les capteurs `S_15 et S_2` bougent mais très peu, ils sont à surveiller mais ne sont pas les plus importants ici. <br>
Après une analyse rigoureuse des visualisations des capteurs restants pour déterminer lesquelles ont une tendance suffisamment claire pour avoir un impact sur la dégradation du moteur, Une question née, on doit désormais savoir si le rapport **signale/bruit** est plus informatif que la **tendance** sur l'impact du capte sur la dégradation du moteur

In [ ]:
resultats = {}

for sensor in sensor_names:
    pente, _, _, _, _ = stats.linregress(moteur1['time_cycles'], moteur1[sensor])
    bruit = moteur1[sensor].std()
    if bruit > 0:
        ratio = abs(pente) / bruit
    else:
        ratio = 0
    resultats[sensor] = ratio

ratios = pd.Series(resultats).sort_values(ascending=False)
ratios


* pente = à quel point la valeur change en moyenne à chaque cycle (la tendance)
* bruit = l'écart-type (la dispersion autour de cette tendance)
* ratio = plus il est élevé, plus le signal de dégradation est clair par rapport au bruit — donc plus le capteur est utile

Le ratio pente/bruit nous permet de voir à quel point le signal de dégradation d'un capteur est clair par rapport à son bruit, indépendamment de sa vitesse de dégradation.

L'analyse des résultats confirme ce qui a été observé visuellement : les capteurs qui n'apportent aucune information restent les mêmes (S_1, S_5, S_6, S_10, S_16, S_18, S_19 — ratio nul).

Pour les 14 capteurs restants, le ratio est assez homogène (entre 0.008 et 0.015) : aucun ne se distingue clairement des autres en termes de clarté du signal de dégradation. Ce calcul a été fait sur un seul moteur (unit 1) — il faudra vérifier si ce classement reste stable sur l'ensemble des 100 moteurs avant de tirer une conclusion définitive.

In [ ]:
resultats_tous_moteurs = []

for unit in train_df['unit_number'].unique() :
  moteur = train_df[train_df['unit_number'] == unit]
  ratio_moteur = {}

  for sensor in sensor_names :
    pente, _, _, _, _= stats.linregress(moteur['time_cycles'], moteur[sensor])
    bruit = moteur[sensor].std()

    if bruit > 0 :
      ratio = abs(pente)/bruit

    else :
      ratio = 0

    ratio_moteur[sensor] = ratio

  resultats_tous_moteurs.append(ratio_moteur)

df_ratios = pd.DataFrame(resultats_tous_moteurs)
df_mean = df_ratios.mean().sort_values(ascending = False)

df_mean

En conclusion, la sélection des 7 capteurs à exclure (capteurs plats) reste stable. Bien que la moyenne des ratios du capteur S_6 sur les 100 moteurs montre qu'il n'est plus totalement plat, son changement est minime, presque négligeable — on le garde donc dans la catégorie des capteurs exclus.

Les écarts entre les ratios des 14 autres capteurs restent assez faibles, on ne constate toujours aucune hiérarchie évidente parmi eux.

## **Etape 2** : Validation statistique avec le test de Mann-kendal

In [ ]:
import pymannkendall as mk
data = {}

for sensor in sensor_names:
    resultat = mk.original_test(moteur1[sensor])
    data[sensor] = [resultat.trend, resultat.p]

trend_df = pd.DataFrame(data).T.rename(columns={0 : 'Trend',1 : 'P.value'})
trend_df.sort_values(ascending=False, by = 'P.value')